In [ ]:
"""
build_ss_datasets.py
====================
Gera 3 tabelas merged para Looker Studio a partir dos CSVs de Segurança Social.

Outputs:
  fact_foreigner_ss_by_district.csv     (grain: year × district)
  fact_foreigner_ss_by_gender_age.csv   (grain: year × gender × age_group)
  fact_total_workers_by_district.csv    (grain: year × district)

Uso:
  python build_ss_datasets.py
  python build_ss_datasets.py --input_dir ./data --output_dir ./output
"""

from pathlib import Path
import pandas as pd


In [23]:
# ── 1. CONFIGURAÇÕES E PATHS ──────────────────────────────────────────────────

# Como o notebook está em notebooks/seg_social/, subimos 2 níveis (parents[1]) até à raiz
REPO_ROOT = Path.cwd().parents[1] if Path.cwd().name == "seg_social" else Path.cwd()

INPUT_DIR             = REPO_ROOT / "data" / "2-clean" / "seg_social"
OUTPUT_INDIVIDUAL_DIR = REPO_ROOT / "data" / "3-delivery" / "seg_social" / "fontes_individuais"
OUTPUT_MERGED_DIR     = REPO_ROOT / "data" / "3-delivery" / "seg_social"

SOURCE_ID   = "F014"
SOURCE_NAME = "Segurança Social"
SEP         = ";"
ENCODING    = "utf-8"

# 🔍 Validação dos caminhos
print(f"📍 Raiz do Projeto: {REPO_ROOT.resolve()}")
print(f"📂 Pasta de Entrada: {INPUT_DIR.resolve()}")
print(f"❓ Pasta de entrada existe? {INPUT_DIR.exists()}")

📍 Raiz do Projeto: C:\zardit\estagio-prepara-portugal
📂 Pasta de Entrada: C:\zardit\estagio-prepara-portugal\data\2-clean\seg_social
❓ Pasta de entrada existe? True


In [24]:
# ──2. Dicionários de Padronização ──────────────────────────────────────────────

DISTRICT_MAP = {
    "AVEIRO": "Aveiro",
    "BEJA": "Beja",
    "BRAGA": "Braga",
    "BRAGANCA": "Bragança",
    "CASTELO BRANCO": "Castelo Branco",
    "COIMBRA": "Coimbra",
    "EVORA": "Évora",
    "FARO": "Faro",
    "GUARDA": "Guarda",
    "LEIRIA": "Leiria",
    "LISBOA": "Lisboa",
    "OUTRO": "Outro",
    "PORTALEGRE": "Portalegre",
    "PORTO": "Porto",
    "RA ACORES": "Região Autónoma dos Açores",
    "RA MADEIRA": "Região Autónoma da Madeira",
    "SANTAREM": "Santarém",
    "SETUBAL": "Setúbal",
    "VIANA DO CASTELO": "Viana do Castelo",
    "VILA REAL": "Vila Real",
    "VISEU": "Viseu",
}

AGE_MAP = {
    "MENOS - 20": "<20",
    "MENOS DE 20": "<20",
    "<20": "<20",
    "MAIS DE 60": "60+",
    "MAIS - 60": "60+",
    ">60": "60+",
    "60+": "60+",
}

In [25]:
# ── 3. FUNÇÕES AUXILIARES DE LIMPEZA E Carga ──────────────────────────────────

def process_and_clean_file(file_path: Path, output_dir: Path) -> pd.DataFrame:
    """
    Lê, padroniza, grava o ficheiro individual limpo com source/source_id,
    e devolve o DataFrame limpo (em memória) sem as colunas de fonte para o merge.
    """
    df = pd.read_csv(file_path, sep=SEP, encoding=ENCODING, dtype=str)
    
    # Trim em colunas de texto
    str_cols = df.select_dtypes(include=["object", "string"]).columns
    for col in str_cols:
        df[col] = df[col].str.strip()

    # 1. Padronizar Distrito
    if "district" in df.columns:
        df["district"] = df["district"].str.upper().replace(DISTRICT_MAP)

    # 2. Padronizar Género (Feminino / Masculino)
    if "gender" in df.columns:
        df["gender"] = df["gender"].str.capitalize()

    # 3. Padronizar Faixa Etária
    for age_col in ["age", "age_group"]:
        if age_col in df.columns:
            df[age_col] = (
                df[age_col]
                .astype(str)
                .str.replace(r"\s+", " ", regex=True)     # Remove múltiplos espaços/espaços invisíveis
                .str.strip()                              # Limpa extremidades
                .str.upper()                              # Garante maiúsculas para o replace
            )

            # Aplica primeiro o mapeamento dos limites ("MENOS DE 20" -> "<20")
            df[age_col] = df[age_col].replace(AGE_MAP)

            # Remove QUALQUER espaço à volta de traços em intervalos (ex: "20 - 29" -> "20-29")
            df[age_col] = df[age_col].str.replace(r"\s*-\s*", "-", regex=True)

    # 4. Converter colunas numéricas para calculo/merge
    num_cols = [c for c in df.columns if c.endswith(("_amount", "_count")) or c in ["net_balance", "year"]]
    for col in num_cols:
        if col == "year":
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")
        else:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Round em montantes
    amount_cols = [c for c in df.columns if c.endswith("_amount") or c == "net_balance"]
    for col in amount_cols:
        df[col] = df[col].round(2)

    # ── Exportação Individual com Source ──────────────────────────────────────
    df_export = df.copy()
    df_export["source_id"] = SOURCE_ID
    df_export["source"] = SOURCE_NAME

    # Reordenar colunas
    cols = list(df_export.columns)
    other_cols = [c for c in cols if c not in ("year", "source_id", "source")]
    new_order = (["year"] if "year" in cols else []) + other_cols + ["source_id", "source"]
    df_export = df_export[new_order]

    output_dir.mkdir(parents=True, exist_ok=True)
    df_export.to_csv(output_dir / file_path.name, sep=SEP, encoding=ENCODING, index=False)

    return df

def validate_merge(df: pd.DataFrame, keys: list[str], label: str):
    """Valida duplicados e imprime resumo."""
    dupes = df.duplicated(subset=keys).sum()
    if dupes > 0:
        print(f"  ⚠️  {label}: {dupes} linha(s) duplicada(s) na chave {keys}")
    else:
        print(f"  ✅ {label}: sem duplicados ({len(df)} linhas, anos {df['year'].min()}–{df['year'].max()})")

In [26]:
# ── 4. CONSTRUÇÃO DOS DATASETS MERGED ─────────────────────────────────────────

def build_all_datasets():
    print(f"📂 Lendo e limpando origens de: {INPUT_DIR.resolve()}")
    print(f"📂 A guardar individuais em:   {OUTPUT_INDIVIDUAL_DIR.resolve()}\n")

    # Mapeamento em memória dos DataFrames limpos
    raw_files = list(INPUT_DIR.rglob("*.csv"))
    if not raw_files:
        raise FileNotFoundError(f"Nenhum ficheiro CSV encontrado em {INPUT_DIR}")

    dfs = {}
    for f in raw_files:
        dfs[f.name] = process_and_clean_file(f, OUTPUT_INDIVIDUAL_DIR)
        print(f"  ➕ Ficheiro individual limpo e salvo: {f.name}")

    # ── Tabela 1: Foreigners Contribution & Benefit by District ───────────────
    print("\n📦 A gerar Tabela 1: ss_foreigners_contribution_benefit_by_district_2015_2025.csv")
    keys_dist = ["year", "district"]
    
    t1 = (
        dfs["ss_foreigner_contribution_amounts_by_district_2015_2025.csv"]
        .merge(dfs["ss_foreigner_benefit_amounts_by_district_2015_2025.csv"], on=keys_dist, how="outer")
        .merge(dfs["ss_foreigner_contributors_by_district_2015_2025.csv"], on=keys_dist, how="outer")
        .merge(dfs["ss_foreigner_beneficiaries_by_district_2015_2025.csv"], on=keys_dist, how="outer")
        .merge(dfs["ss_foreigner_new_registrations_by_district_2015_2025.csv"], on=keys_dist, how="outer")
        .sort_values(keys_dist)
        .reset_index(drop=True)
    )

    t1["net_balance"] = (t1["contribution_amount"] - t1["benefit_amount"]).round(2)
    t1["avg_contribution_per_contributor"] = (t1["contribution_amount"] / t1["contributors_count"]).round(2)
    t1["avg_benefit_per_beneficiary"] = (t1["benefit_amount"] / t1["beneficiaries_count"]).round(2)
    t1["contribution_benefit_ratio"] = (t1["contribution_amount"] / t1["benefit_amount"]).round(4)

    validate_merge(t1, keys_dist, "Tabela 1 (District)")

    # ── Tabela 2: Foreigners Contributors & Beneficiaries by Gender & Age ─────
    print("\n📦 A gerar Tabela 2: ss_foreigners_contributors_beneficiaries_by_gender_age_2015_2025.csv")
    keys_ga = ["year", "gender", "age_group"]

    t2 = (
        dfs["ss_foreigner_contributors_by_gender_age_group_2015_2025.csv"]
        .merge(dfs["ss_foreigner_beneficiaries_by_gender_age_group_2015_2025.csv"], on=keys_ga, how="outer")
        .merge(dfs["ss_foreigner_new_registrations_by_gender_age_group_2015_2025.csv"], on=keys_ga, how="outer")
        .sort_values(keys_ga)
        .reset_index(drop=True)
    )

    validate_merge(t2, keys_ga, "Tabela 2 (Gender/Age)")

    # ── Tabela 3: Total Workers Context by District ───────────────────────────
    print("\n📦 A gerar Tabela 3: ss_total_workers_context_by_district_2010_2025.csv")

    emp_contrib = dfs["ss_employee_contribution_amounts_by_district_2010_2025.csv"].rename(
        columns={"contribution_amount": "employee_contribution_amount"}
    )
    se_contrib = dfs["ss_self_employed_contribution_amounts_by_district_2010_2025.csv"].rename(
        columns={"contribution_amount": "self_employed_contribution_amount"}
    )

    t3 = (
        dfs["ss_employees_by_district_2010_2025.csv"]
        .merge(dfs["ss_self_employed_by_district_2010_2025.csv"], on=keys_dist, how="outer")
        .merge(emp_contrib, on=keys_dist, how="outer")
        .merge(se_contrib, on=keys_dist, how="outer")
        .merge(dfs["ss_employee_remuneration_amounts_by_district_2010_2025.csv"], on=keys_dist, how="outer")
        .merge(dfs["ss_employers_base_remuneration_by_district_2010_2025.csv"], on=keys_dist, how="outer")
        .sort_values(keys_dist)
        .reset_index(drop=True)
    )

    t3["total_workers"] = t3["employees_count"] + t3["self_employed_count"]
    t3["total_contribution_amount"] = (
        t3["employee_contribution_amount"] + t3["self_employed_contribution_amount"]
    ).round(2)

    validate_merge(t3, keys_dist, "Tabela 3 (Total Workers)")

    # ══════════════════════════════════════════════════════════════════════════
    # 👇 AQUI ENTRA O PRIMEIRO TRECHO (DENTRO DA FUNÇÃO, COM 4 ESPAÇOS DE INDENTAÇÃO)
    # ══════════════════════════════════════════════════════════════════════════
    for df in [t1, t2, t3]:
        df["source_id"] = SOURCE_ID
        df["source"] = SOURCE_NAME

    # Reordenar colunas (year no início, source no fim)
    t1 = t1[["year"] + [c for c in t1.columns if c not in ("year", "source_id", "source")] + ["source_id", "source"]]
    t2 = t2[["year"] + [c for c in t2.columns if c not in ("year", "source_id", "source")] + ["source_id", "source"]]
    t3 = t3[["year"] + [c for c in t3.columns if c not in ("year", "source_id", "source")] + ["source_id", "source"]]

    return t1, t2, t3

In [27]:
# ── Executa a função e guarda os DataFrames em t1, t2 e t3 ────────────────────
t1, t2, t3 = build_all_datasets()

# ── ESCRITA DOS FICHEIROS MERGED NO DESTINO FINAL ─────────────────────────────
OUTPUT_MERGED_DIR.mkdir(parents=True, exist_ok=True)

t1.to_csv(OUTPUT_MERGED_DIR / "ss_foreigners_contribution_benefit_by_district_2015_2025.csv", index=False, sep=SEP, encoding=ENCODING)
t2.to_csv(OUTPUT_MERGED_DIR / "ss_foreigners_contributors_beneficiaries_by_gender_age_2015_2025.csv", index=False, sep=SEP, encoding=ENCODING)
t3.to_csv(OUTPUT_MERGED_DIR / "ss_total_workers_context_by_district_2010_2025.csv", index=False, sep=SEP, encoding=ENCODING)

print("\n🎉 Ficheiros salvos com sucesso em 3-delivery/seg_social!")

📂 Lendo e limpando origens de: C:\zardit\estagio-prepara-portugal\data\2-clean\seg_social
📂 A guardar individuais em:   C:\zardit\estagio-prepara-portugal\data\3-delivery\seg_social\fontes_individuais

  ➕ Ficheiro individual limpo e salvo: ss_employees_by_district_2010_2025.csv
  ➕ Ficheiro individual limpo e salvo: ss_employee_contribution_amounts_by_district_2010_2025.csv
  ➕ Ficheiro individual limpo e salvo: ss_employee_remuneration_amounts_by_district_2010_2025.csv
  ➕ Ficheiro individual limpo e salvo: ss_employers_base_remuneration_by_district_2010_2025.csv
  ➕ Ficheiro individual limpo e salvo: ss_foreigner_beneficiaries_by_district_2015_2025.csv
  ➕ Ficheiro individual limpo e salvo: ss_foreigner_beneficiaries_by_gender_age_group_2015_2025.csv
  ➕ Ficheiro individual limpo e salvo: ss_foreigner_benefit_amounts_by_district_2015_2025.csv
  ➕ Ficheiro individual limpo e salvo: ss_foreigner_contribution_amounts_by_district_2015_2025.csv
  ➕ Ficheiro individual limpo e salvo: ss_f

In [ ]:
# Executar
build_all_datasets()